#Training U-Net และ Pretrained Model สำหรับ Semantic Segmentation

### 🎯 สิ่งที่จะได้เรียนรู้จาก Notebook นี้

Notebook นี้ถูกออกแบบมาเพื่อให้ผู้เรียนได้ลงมือทำ (Hands-on) ในงาน **Semantic Segmentation** แบบครบวงจร ตั้งแต่การเตรียมข้อมูลไปจนถึงการนำโมเดลไปใช้งานจริง โดยมีหัวข้อหลักดังนี้:

1. **การเตรียมข้อมูลและ Data Augmentation:**
   - การโหลดและจัดการชุดข้อมูล Pascal VOC 2012 (ภาพและหน้ากากแบ่งส่วน)
   - การใช้ไลบรารี `albumentations` เพื่อทำ Data Augmentation เพิ่มความหลากหลายให้ข้อมูลฝึกสอน (Training Data)

2. **การสร้างและฝึกสอนโมเดล Deep Learning 2 สถาปัตยกรรม:**
   - **U-Net:** การใช้ไลบรารี `segmentation_models_pytorch` เพื่อสร้างโมเดล U-Net แบบรวดเร็ว โดยใช้ ResNet34 เป็น Encoder
   - **SegFormer:** การใช้ไลบรารี `transformers` จาก Hugging Face เพื่อนำโมเดลกลุ่ม Vision Transformer (mit-b0) มา Fine-tune

3. **การเขียน Training Loop และการวัดผล (Evaluation):**
   - การเขียนลูปฝึกสอนโมเดลด้วย PyTorch (Train/Validation/Test)
   - การคำนวณมาตรวัด **mIoU (Mean Intersection over Union)** ซึ่งเป็นมาตรวัดมาตรฐานสำหรับงาน Segmentation

4. **การแสดงผลลัพธ์ภาพ (Visualization):**
   - การเปรียบเทียบภาพต้นฉบับ (Original), เฉลย (Ground Truth) และผลลัพธ์จากการทำนาย (Prediction)

5. **การสร้าง Interactive Web App ด้วย Gradio:**
   - การนำโมเดล U-Net และ SegFormer ที่เทรนเสร็จแล้ว มาสร้าง Web App ให้ผู้ใช้สามารถอัปโหลดรูปภาพของตนเองมาทดสอบได้ทันที พร้อม UI เปรียบเทียบผลลัพธ์ที่สวยงาม

## 1. Setup & Imports (ติดตั้งและนำเข้าไลบรารี)
ติดตั้งไลบรารีที่จำเป็นสำหรับ Semantic Segmentation เช่น `segmentation_models_pytorch` และนำเข้าโมดูลต่างๆ

In [ ]:
!pip install datasets segmentation-models-pytorch albumentations torch torchvision tqdm -q

## 2. Dataset & Data Augmentation (เตรียมชุดข้อมูล)
ในขั้นตอนนี้เราจะใช้งานชุดข้อมูล **Pascal VOC 2012** ซึ่งเป็นชุดข้อมูลมาตรฐานระดับโลกสำหรับงาน Computer Vision โดยในส่วนของ Semantic Segmentation จะประกอบไปด้วย **21 คลาส** (สิ่งของ 20 ชนิด เช่น คน, รถ, สัตว์ และ พื้นหลังอีก 1 คลาส) ซึ่งภาพแต่ละภาพจะมาพร้อมกับ Mask ที่ระบุคลาสของแต่ละพิกเซล
นอกจากนี้ เราจะสร้าง Custom Dataset สำหรับโหลดรูปภาพและ Masks พร้อมทั้งทำ Data Augmentation เพื่อเพิ่มความหลากหลายให้ข้อมูล

In [ ]:
import torchvision
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split
import os

# โหลดข้อมูลชุด Train จาก Pascal VOC (จะใช้เวลาดาวน์โหลดในครั้งแรก)
dataset = torchvision.datasets.VOCSegmentation(root='./data', year='2012', image_set='train', download=True)

print("Dataset size:", len(dataset))

# ดูตัวอย่างข้อมูล
image, mask = dataset[0]

print(f"Image size : {image.size}")
print(f"Mask  size : {mask.size}")
print(f"Mask  mode : {mask.mode}")

# Pascal VOC มี 21 คลาส (0 = พื้นหลัง, 255 = ขอบเขตที่ละเว้น)
VOC_CLASSES = [
    "background", "aeroplane", "bicycle", "bird", "boat", "bottle",
    "bus", "car", "cat", "chair", "cow", "diningtable", "dog",
    "horse", "motorbike", "person", "pottedplant", "sheep", "sofa",
    "train", "tvmonitor",
]
NUM_CLASSES = len(VOC_CLASSES)
print(f"Number of classes: {NUM_CLASSES}")

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

IMG_SIZE   = 256
VOID_LABEL = 255   # ขอบเขตภาพที่ไม่ต้องนำมาคิด

# ==========================================
# ส่วนที่ 1: Data Augmentation (การเพิ่มความหลากหลายให้ข้อมูล)
# ==========================================
train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE), # ปรับขนาดภาพให้เท่ากัน
    A.HorizontalFlip(p=0.5),      # สุ่มพลิกภาพซ้าย-ขวา
    A.RandomBrightnessContrast(p=0.3), # สุ่มปรับความสว่างและความคมชัด
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.4), # สุ่มเลื่อน หมุน หรือซูมภาพ
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)), # ปรับค่าสีให้อยู่ในมาตรฐานเดียวกับโมเดล
    ToTensorV2(), # แปลงภาพเป็น Tensor เพื่อใช้ใน PyTorch
])

# สำหรับชุดทดสอบ (Validation) เราแค่ปรับขนาดและปรับค่าสี ไม่ต้องสุ่มบิดเบือนภาพ
val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])


In [ ]:
# ==========================================
# ส่วนที่ 2: Data Processing & DataLoader
# ==========================================
val_raw = torchvision.datasets.VOCSegmentation(
    root='./data', year='2012', image_set='val', download=True
)

class VOCDataset(Dataset):
    def __init__(self, torchvision_dataset, transform=None):
        self.data = torchvision_dataset
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image, mask = self.data[idx]

        image = np.array(image.convert("RGB"))
        mask = np.array(mask.convert("P"), dtype=np.uint8)

        mask[mask == VOID_LABEL] = 0

        if self.transform:
            aug = self.transform(image=image, mask=mask)
            image = aug["image"]
            mask = aug["mask"].long()

        return image, mask

# แยก validation ออกเป็น val/test
VAL_RATIO = 0.5
val_size = int(len(val_raw) * VAL_RATIO)
test_size = len(val_raw) - val_size

val_subset, test_subset = random_split(
    val_raw,
    [val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

train_dataset = VOCDataset(dataset, train_transform)
val_dataset = VOCDataset(val_subset, val_transform)
test_dataset = VOCDataset(test_subset, val_transform)

BATCH_SIZE = 8

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True
)

val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True
)

test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True
)

print(f"จำนวนข้อมูล Train : {len(train_dataset)} รูป")
print(f"จำนวนข้อมูล Val   : {len(val_dataset)} รูป")
print(f"จำนวนข้อมูล Test  : {len(test_dataset)} รูป")

imgs, msks = next(iter(train_loader))
print(f"\nขนาดภาพ 1 Batch : {imgs.shape} -> (Batch, Channel, Height, Width)")
print(f"ขนาด Mask 1 Batch : {msks.shape} -> (Batch, Height, Width)")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ดึงข้อมูลจาก DataLoader มา 1 Batch เพื่อแสดงผล
imgs, msks = next(iter(train_loader))

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
cmap = plt.get_cmap("tab20")

present_classes = set()

for i in range(4):
    # แปลงภาพกลับให้อยู่ในช่วง 0-1 เพื่อแสดงผลปกติ (Denormalize)
    img = imgs[i].permute(1, 2, 0).cpu().numpy()
    img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    img = np.clip(img, 0, 1)

    mask = msks[i].cpu().numpy()
    present_classes.update(np.unique(mask))

    # แถวบน: ภาพต้นฉบับ
    axes[0, i].imshow(img)
    axes[0, i].set_title(f"Image {i+1}")
    axes[0, i].axis("off")

    # แถวล่าง: ภาพซ้อนทับ Mask (Overlay)
    axes[1, i].imshow(img)
    axes[1, i].imshow(mask, cmap=cmap, vmin=0, vmax=20, alpha=0.8)
    axes[1, i].set_title(f"Ground Truth {i+1}")
    axes[1, i].axis("off")

# เพิ่ม Legend อธิบายคลาส (เฉพาะที่มีในภาพ)
legend_elements = [mpatches.Patch(color=cmap(c/20.0), label=VOC_CLASSES[c]) for c in sorted(present_classes) if c < NUM_CLASSES]
fig.legend(handles=legend_elements, loc='center right', bbox_to_anchor=(1.12, 0.5), title="Classes")

plt.tight_layout()
plt.subplots_adjust(right=0.88)
plt.show()

## 3. สร้างโมเดล U-Net
ในส่วนนี้เราจะสร้างโมเดล U-Net จากไลบรารี `segmentation_models_pytorch` โดยใช้ Encoder เป็น `resnet34` ซึ่งโหลด Pre-trained weights จาก ImageNet กำหนดจำนวนคลาสให้ตรงกับข้อมูล และนำโมเดลไปไว้ที่ GPU จากนั้นจะทดสอบรันด้วยข้อมูลจำลอง (Dummy Input) เพื่อตรวจสอบขนาดของ Output

In [ ]:
import segmentation_models_pytorch as smp

ENCODER         = 'resnet34'
ENCODER_WEIGHTS = 'imagenet'
CLASSES         = NUM_CLASSES
ACTIVATION      = None

# สร้างโมเดล U-Net
model = smp.Unet(
    encoder_name    = ENCODER,
    encoder_weights = ENCODER_WEIGHTS,
    classes         = CLASSES,
    activation      = ACTIVATION,
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")
model = model.to(DEVICE)

# ทดสอบให้ข้อมูล dummy ผ่านโมเดล
dummy_in  = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
dummy_out = model(dummy_in)
print(f"Input  shape: {dummy_in.shape}")
print(f"Output shape: {dummy_out.shape}")

## 4. Loss, Optimizer & Training Loop (กำหนดฟังก์ชันความสูญเสียและเริ่มเทรน)
เราจะใช้ `CrossEntropyLoss` เป็น Loss Function โดยข้ามการคิด loss ในส่วนของขอบเขตที่ไม่สนใจ (VOID_LABEL) และใช้ `AdamW` optimizer ร่วมกับตัวปรับลด Learning Rate แบบ `CosineAnnealingLR`

นอกจากนี้ได้สร้างฟังก์ชันคำนวณมาตรวัด **mIoU** (Mean Intersection over Union) และเขียนลูปสำหรับ Train/Validate โดยจะประเมินผลทุก Epoch และบันทึกโมเดลที่มีค่า Validation mIoU สูงสุดเก็บไว้ (`best_unet_resnet34.pth`)

In [ ]:
from torch import nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import torch.nn.functional as F

criterion = nn.CrossEntropyLoss(ignore_index=VOID_LABEL)

optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

EPOCHS    = 40
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

# ฟังก์ชันสำหรับคำนวณ mIoU
def get_logits(outputs):
    if hasattr(outputs, "logits"):
        return outputs.logits
    return outputs

def mean_iou(preds, targets, num_classes=NUM_CLASSES, ignore_index=255):
    logits = get_logits(preds)

    # resize logits ให้เท่ากับ mask ถ้าขนาดไม่ตรงกัน
    if logits.shape[-2:] != targets.shape[-2:]:
        logits = F.interpolate(
            logits,
            size=targets.shape[-2:],
            mode="bilinear",
            align_corners=False
        )

    preds = logits.argmax(dim=1).cpu().numpy().ravel()
    targets = targets.cpu().numpy().ravel()

    valid = targets != ignore_index
    preds, targets = preds[valid], targets[valid]

    ious = []
    for c in range(num_classes):
        inter = ((preds == c) & (targets == c)).sum()
        union = ((preds == c) | (targets == c)).sum()

        if union > 0:
            ious.append(inter / union)

    return float(np.mean(ious)) if ious else 0.0

In [ ]:
from tqdm import tqdm

best_miou   = 0.0
history     = {"train_loss": [], "val_loss": [], "val_miou": []}

for epoch in range(1, EPOCHS + 1):
    # ── (Train) ──
    model.train()
    train_loss = 0.0

    for images, masks in tqdm(train_loader, desc=f"Epoch {epoch:02d}/{EPOCHS} [train]"):
        images, masks = images.to(DEVICE), masks.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)

    train_loss /= len(train_dataset)

    # ── (Validate) ──
    model.eval()
    val_loss, val_miou = 0.0, 0.0

    with torch.no_grad():
        for images, masks in tqdm(val_loader, desc=f"Epoch {epoch:02d}/{EPOCHS} [val]  "):
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            outputs  = model(images)
            val_loss += criterion(outputs, masks).item() * images.size(0)
            val_miou += mean_iou(outputs, masks) * images.size(0)

    val_loss  /= len(val_dataset)
    val_miou  /= len(val_dataset)

    scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_miou"].append(val_miou)

    print(f"Epoch {epoch:02d}/{EPOCHS}  |  "
          f"Train Loss: {train_loss:.4f}  |  "
          f"Val Loss: {val_loss:.4f}  |  "
          f"Val mIoU: {val_miou:.4f}")

    # บันทึกโมเดลที่ดีที่สุด
    if val_miou > best_miou:
        best_miou = val_miou
        torch.save(model.state_dict(), "best_unet_resnet34.pth")
        print(f"  ✅ Saved best model (mIoU = {best_miou:.4f})")

print(f"\nTraining complete. Best Val mIoU: {best_miou:.4f}")

## 5. Evaluation & Visualization (ประเมินผลและแสดงภาพ)
ในส่วนนี้เราจะเริ่มต้นด้วยการพล็อตกราฟ Training/Validation Loss และค่า mIoU เพื่อดูพฤติกรรมการเรียนรู้ของโมเดล

จากนั้นจะทำการโหลด U-Net โมเดลที่ดีที่สุดมาประเมินค่า mIoU สุดท้ายบน **Test Set** และเรียกใช้ฟังก์ชัน `visualize_predictions` ที่เราเตรียมไว้ เพื่อวาดภาพเปรียบเทียบระหว่างภาพต้นฉบับ, Ground Truth และ Prediction พร้อมแสดง Legend อธิบายคลาสที่ปรากฏในภาพ

In [ ]:
import matplotlib.pyplot as plt

def plot_training_curves(history, save_path="training_curves.png"):
    """
    Plot training/validation loss and validation mIoU.

    Args:
        history (dict): {
            "train_loss": [...],
            "val_loss": [...],
            "val_miou": [...]
        }
        save_path (str): path to save the figure
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # Loss plot
    axes[0].plot(history["train_loss"], label="Train Loss")
    axes[0].plot(history["val_loss"], label="Val Loss")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()

    # mIoU plot
    axes[1].plot(history["val_miou"], label="Val mIoU")
    axes[1].set_title("Validation mIoU")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("mIoU")
    axes[1].legend()

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150)

    plt.show()

plot_training_curves(history, save_path="unet_training.png")

จะเห็นได้ว่าโมเดล U-Net เริ่มเกิด overfitting โดยแม้จะฝึกนานขึ้น ค่า validation loss เริ่มคงที่ และค่า validation mIoU ก็เริ่มเข้าสู่ภาวะ plateau ไม่ได้เพิ่มขึ้น

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tqdm import tqdm

# โหลดโมเดล
unet_model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights=None,
    in_channels=3,
    classes=NUM_CLASSES
).to(DEVICE)

unet_model.load_state_dict(
    torch.load("best_unet_resnet34.pth", map_location=DEVICE)
)

unet_model.eval()

# ประเมิน test
test_miou = 0.0
with torch.no_grad():
    for images, masks in tqdm(test_loader):
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        outputs = unet_model(images)
        test_miou += mean_iou(outputs, masks) * images.size(0)

test_miou /= len(test_dataset)
print(f"\nTest mIoU: {test_miou:.4f}\n")

# ฟังก์ชัน predict
def predict(model, pil_image, device=DEVICE):
    model.eval()

    img_t = val_transform(image=np.array(pil_image.convert("RGB")))["image"]
    img_t = img_t.unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(img_t)
        logits = get_logits(outputs)

        # resize กลับเท่าภาพ input ถ้าจำเป็น
        if logits.shape[-2:] != img_t.shape[-2:]:
            logits = F.interpolate(
                logits,
                size=img_t.shape[-2:],
                mode="bilinear",
                align_corners=False
            )

        pred = logits.argmax(dim=1)

    return pred.squeeze().cpu().numpy()

# ฟังก์ชันสำหรับพล็อตภาพผลลัพธ์
def visualize_predictions(eval_model, dataset, num_samples=4):
    idxs = np.random.choice(len(dataset), size=num_samples, replace=False)

    fig, axes = plt.subplots(num_samples, 3, figsize=(10, 3 * num_samples))
    cmap = plt.get_cmap("tab20")
    present_classes = set()

    for row, idx in enumerate(idxs):
        img_tensor, gt_mask = dataset[idx]
        pil_image, _ = dataset.data[idx]

        # แปลงภาพ
        img = img_tensor.permute(1, 2, 0).cpu().numpy()
        img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        img = np.clip(img, 0, 1)

        gt_mask = gt_mask.cpu().numpy()
        pred_mask = predict(eval_model, pil_image)

        # เก็บค่าคลาสที่พบ
        present_classes.update(np.unique(gt_mask))
        present_classes.update(np.unique(pred_mask))

        # แสดงผล
        axes[row, 0].imshow(img)
        if row == 0: axes[row, 0].set_title("Original")
        axes[row, 0].axis("off")

        axes[row, 1].imshow(img)
        axes[row, 1].imshow(
            np.ma.masked_where(gt_mask == 0, gt_mask),
            cmap=cmap, alpha=0.6, vmin=0, vmax=20
        )
        if row == 0: axes[row, 1].set_title("Ground Truth")
        axes[row, 1].axis("off")

        axes[row, 2].imshow(img)
        axes[row, 2].imshow(
            np.ma.masked_where(pred_mask == 0, pred_mask),
            cmap=cmap, alpha=0.6, vmin=0, vmax=20
        )
        if row == 0: axes[row, 2].set_title("Prediction")
        axes[row, 2].axis("off")

    # เพิ่ม Legend อธิบายคลาส (แสดงเฉพาะคลาสที่มีอยู่จริง)
    legend_elements = [mpatches.Patch(color=cmap(c/20.0), label=VOC_CLASSES[c]) for c in sorted(present_classes) if c < NUM_CLASSES]
    fig.legend(handles=legend_elements, loc='center right', bbox_to_anchor=(1.2, 0.5), title="Present Classes")

    plt.tight_layout()
    plt.subplots_adjust(right=0.85)
    plt.show()

# เรียกใช้งานสำหรับ U-Net
visualize_predictions(unet_model, test_dataset)


## 6. สร้างและ Train SegFormer
ถัดมาเราจะทดลองใช้สถาปัตยกรรม **SegFormer** ซึ่งเป็นโมเดลสมัยใหม่ที่ใช้ **Vision Transformer (ViT)** แทน Convolutional Neural Network (CNN) แบบดั้งเดิม โดยเราจะใช้โมเดล Pre-trained รุ่น `mit-b0` จากไลบรารี `transformers` (Hugging Face) มา Fine-tune ให้เข้ากับชุดข้อมูล Pascal VOC ของเรา

In [ ]:
from transformers import AutoModelForSemanticSegmentation
import torch.nn.functional as F

CHECKPOINT = "nvidia/mit-b0"
EPOCHS     = 40
LR         = 1e-4
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'

VOC_CLASSES = [
    "background", "aeroplane", "bicycle", "bird", "boat", "bottle",
    "bus", "car", "cat", "chair", "cow", "diningtable", "dog",
    "horse", "motorbike", "person", "pottedplant", "sheep", "sofa",
    "train", "tvmonitor",
]
id2label    = {i: c for i, c in enumerate(VOC_CLASSES)}
label2id    = {c: i for i, c in enumerate(VOC_CLASSES)}

# ==========================================
# Model
# ==========================================
segformer_model = AutoModelForSemanticSegmentation.from_pretrained(
    CHECKPOINT,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
).to(DEVICE)

# ==========================================
# Loss / Optimizer / Scheduler — เหมือน UNet ทุกบรรทัด ยกเว้น layer LR
# ==========================================
segformer_criterion = nn.CrossEntropyLoss(ignore_index=VOID_LABEL)

segformer_optimizer = AdamW([
    {"params": segformer_model.segformer.parameters(),    "lr": LR * 0.1},  # encoder pretrained → lr เล็กลง
    {"params": segformer_model.decode_head.parameters(),  "lr": LR},        # decoder → lr เต็ม
], weight_decay=1e-4)                    # เท่ากัน

segformer_scheduler = CosineAnnealingLR(segformer_optimizer, T_max=EPOCHS, eta_min=1e-6)  # เท่ากัน

# ==========================================
# Training Loop — โครงสร้างเหมือน UNet บรรทัดต่อบรรทัด
# ต่างแค่ 2 จุด: (1) upsample logits, (2) save_pretrained
# ==========================================
best_miou = 0.0
history   = {"train_loss": [], "val_loss": [], "val_miou": []}

for epoch in range(1, EPOCHS + 1):

    # ── เริ่มฝึกสอน (Train) ──
    segformer_model.train()
    train_loss = 0.0

    for images, masks in tqdm(train_loader, desc=f"Epoch {epoch:02d}/{EPOCHS} [train]"):
        images, masks = images.to(DEVICE), masks.to(DEVICE)

        segformer_optimizer.zero_grad()
        outputs = segformer_model(pixel_values=images).logits          # (B, 21, H/4, W/4)
        outputs = F.interpolate(outputs, size=masks.shape[-2:],        # → (B, 21, 256, 256)
                                mode="bilinear", align_corners=False)
        loss = segformer_criterion(outputs, masks)
        loss.backward()
        segformer_optimizer.step()

        train_loss += loss.item() * images.size(0)

    train_loss /= len(train_dataset)

    # ── เริ่มทดสอบ (Validate) ──
    segformer_model.eval()
    val_loss, val_miou = 0.0, 0.0

    with torch.no_grad():
        for images, masks in tqdm(val_loader, desc=f"Epoch {epoch:02d}/{EPOCHS} [val]  "):
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            outputs  = segformer_model(pixel_values=images).logits
            outputs  = F.interpolate(outputs, size=masks.shape[-2:],
                                     mode="bilinear", align_corners=False)
            val_loss += segformer_criterion(outputs, masks).item() * images.size(0)
            val_miou += mean_iou(outputs, masks) * images.size(0)

    val_loss /= len(val_dataset)
    val_miou /= len(val_dataset)

    segformer_scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_miou"].append(val_miou)

    print(f"Epoch {epoch:02d}/{EPOCHS}  |  "
          f"Train Loss: {train_loss:.4f}  |  "
          f"Val Loss: {val_loss:.4f}  |  "
          f"Val mIoU: {val_miou:.4f}")

    if val_miou > best_miou:
        best_miou = val_miou
        segformer_model.save_pretrained("./best_segformer_voc")        # HF format แทน torch.save
        print(f"  ✅ Saved best model (mIoU = {best_miou:.4f})")

print(f"\nTraining complete. Best Val mIoU: {best_miou:.4f}")

In [ ]:
plot_training_curves(history, save_path="segformer_training.png")

In [ ]:
from transformers import AutoModelForSemanticSegmentation, AutoImageProcessor
import matplotlib.patches as mpatches

# โหลดโมเดล
BEST_DIR = "/content/best_segformer_voc"
CHECKPOINT = "nvidia/mit-b0"

processor = AutoImageProcessor.from_pretrained(CHECKPOINT)

model = AutoModelForSemanticSegmentation.from_pretrained(
    BEST_DIR,local_files_only=True
).to(DEVICE)

model.eval()

# ประเมิน test
test_miou = 0.0
with torch.no_grad():
    for images, masks in tqdm(test_loader):
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        outputs = model(images)
        test_miou += mean_iou(outputs, masks) * images.size(0)

test_miou /= len(test_dataset)
print(f"\nTest mIoU: {test_miou:.4f}\n")

# เรียกใช้งานฟังก์ชันจากที่สร้างไว้ในส่วนของ U-Net
visualize_predictions(model, test_dataset)


## 7. Interactive Web App ด้วย Gradio
ในส่วนสุดท้ายนี้ เราจะสร้าง Web Application อย่างง่ายด้วยไลบรารี **Gradio** เพื่อนำโมเดล U-Net และ SegFormer ที่ฝึกสอนเสร็จแล้วมาประยุกต์ใช้งานจริง

โดยแอปนี้จะเปิดให้ผู้ใช้สามารถอัปโหลดรูปภาพที่ต้องการทดสอบได้โดยตรง จากนั้นระบบจะทำนายและแสดงผลลัพธ์เปรียบเทียบระหว่างภาพต้นฉบับ กับ Mask จากทั้งสองโมเดล โดยสามารถใช้ Interactive Slider ที่สามารถเลื่อนดูผลลัพธ์แบบ Before/After ได้อย่างชัดเจน

In [ ]:
!pip install --force-reinstall --upgrade gradio


In [ ]:
# ============================================================
# Semantic Segmentation Comparison — U-Net vs SegFormer
# Preserve Original Aspect Ratio in Output
# Google Colab + Gradio 3.50.2
# ============================================================

import os, uuid, torch, numpy as np, base64, io
from PIL import Image
import torch.nn.functional as F
import segmentation_models_pytorch as smp
from transformers import AutoModelForSemanticSegmentation
import albumentations as A
from albumentations.pytorch import ToTensorV2
import gradio as gr

# ─── Pascal VOC ──────────────────────────────────────────────
VOC_CLASSES = [
    "background","aeroplane","bicycle","bird","boat","bottle",
    "bus","car","cat","chair","cow","diningtable","dog",
    "horse","motorbike","person","pottedplant","sheep","sofa",
    "train","tvmonitor",
]

VOC_PALETTE = np.array([
    [0,0,0],[128,0,0],[0,128,0],[128,128,0],[0,0,128],[128,0,128],
    [0,128,128],[128,128,128],[64,0,0],[192,0,0],[64,128,0],[192,128,0],
    [64,0,128],[192,0,128],[64,128,128],[192,128,128],[0,64,0],[128,64,0],
    [0,192,0],[128,192,0],[0,64,128],
], dtype=np.uint8)

NUM_CLASSES = len(VOC_CLASSES)
id2label = {i: c for i, c in enumerate(VOC_CLASSES)}
label2id = {c: i for i, c in enumerate(VOC_CLASSES)}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE = 512

# ─── Preprocessing for model only ────────────────────────────
preprocess = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

def preprocess_image(pil_img):
    arr = np.array(pil_img.convert("RGB"))
    return preprocess(image=arr)["image"].unsqueeze(0).to(DEVICE)

# ─── Load models ─────────────────────────────────────────────
UNET_PATH = "/content/best_unet_resnet34.pth"
SEGFORMER_PATH = "/content/best_segformer_voc"

unet_model = None
segformer_model = None

if os.path.exists(UNET_PATH):
    unet_model = smp.Unet(
        encoder_name="resnet34",
        encoder_weights=None,
        classes=NUM_CLASSES,
        activation=None,
    ).to(DEVICE)
    unet_model.load_state_dict(torch.load(UNET_PATH, map_location=DEVICE))
    unet_model.eval()
    print("✅ U-Net loaded")
else:
    print(f"⚠️ U-Net not found at {UNET_PATH}")

if os.path.exists(SEGFORMER_PATH):
    segformer_model = AutoModelForSemanticSegmentation.from_pretrained(
        SEGFORMER_PATH,
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True,
    ).to(DEVICE)
    segformer_model.eval()
    print("✅ SegFormer loaded")
else:
    print(f"⚠️ SegFormer not found at {SEGFORMER_PATH}")

# ─── Inference helpers ───────────────────────────────────────
def run_unet(tensor):
    with torch.no_grad():
        logits = unet_model(tensor)
    return logits.argmax(dim=1).squeeze().cpu().numpy()

def run_segformer(tensor):
    with torch.no_grad():
        logits = segformer_model(pixel_values=tensor).logits
        logits = F.interpolate(
            logits,
            size=(IMG_SIZE, IMG_SIZE),
            mode="bilinear",
            align_corners=False,
        )
    return logits.argmax(dim=1).squeeze().cpu().numpy()

def mask_to_color(pred):
    color = np.zeros((*pred.shape, 3), dtype=np.uint8)
    for c in range(NUM_CLASSES):
        color[pred == c] = VOC_PALETTE[c]
    return color

def resize_mask_to_original(pred, original_size):
    """
    Convert class prediction to color mask and resize it back
    to original image size without smoothing class colors.
    """
    color_mask = mask_to_color(pred)
    mask_img = Image.fromarray(color_mask)
    mask_img = mask_img.resize(original_size, Image.NEAREST)
    return np.array(mask_img)

def make_overlay(orig_pil, color_mask, alpha=0.6):
    """
    Blend colored mask over original image.
    Original image aspect ratio is preserved.
    """
    base = np.array(orig_pil.convert("RGB"))

    if base.shape[:2] != color_mask.shape[:2]:
        raise ValueError(
            f"Image and mask size mismatch: image={base.shape}, mask={color_mask.shape}"
        )

    blended = (base * (1 - alpha) + color_mask * alpha).astype(np.uint8)
    return Image.fromarray(blended)

def pil_b64(img):
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode()

# ─── Before/After widget ─────────────────────────────────────
def ba_widget(orig_b64, overlay_b64):
    uid = uuid.uuid4().hex[:8]
    return f"""
<div id="wrap_{uid}"
     style="position:relative;display:block;border-radius:10px;
            overflow:hidden;box-shadow:0 4px 20px rgba(0,0,0,.5)">

  <!-- bottom layer: mask overlay, always full width, never touched -->
  <img src="data:image/png;base64,{overlay_b64}"
       style="width:100%;display:block"/>

  <!-- top layer: original, same size, clipped by clip-path only -->
  <!-- clip-path inset(0 0 0 X%) hides left X%, reveals right (100-X)% -->
  <!-- so slider at 50 → left=mask, right=original -->
  <img id="top_{uid}"
       src="data:image/png;base64,{orig_b64}"
       style="position:absolute;top:0;left:0;width:100%;height:100%;
              clip-path:inset(0 0 0 50%);
              object-fit:fill"/>

  <!-- divider line -->
  <div id="line_{uid}"
       style="position:absolute;top:0;bottom:0;left:50%;
              width:3px;margin-left:-1.5px;
              background:rgba(255,255,255,.9);
              box-shadow:0 0 8px rgba(0,0,0,.6);
              pointer-events:none"></div>

  <!-- handle -->
  <div id="hdl_{uid}"
       style="position:absolute;top:50%;left:50%;
              transform:translate(-50%,-50%);
              width:36px;height:36px;border-radius:50%;
              background:#fff;display:flex;align-items:center;
              justify-content:center;font-size:14px;font-weight:900;
              color:#1a1a2e;box-shadow:0 2px 10px rgba(0,0,0,.5);
              pointer-events:none">⇔</div>

  <!-- labels -->
  <span style="position:absolute;bottom:8px;left:8px;font-size:9px;
               font-weight:700;text-transform:uppercase;letter-spacing:.08em;
               padding:2px 7px;border-radius:4px;
               background:rgba(0,0,0,.6);color:#fff;pointer-events:none">
    Mask
  </span>
  <span style="position:absolute;bottom:8px;right:8px;font-size:9px;
               font-weight:700;text-transform:uppercase;letter-spacing:.08em;
               padding:2px 7px;border-radius:4px;
               background:rgba(0,0,0,.6);color:#fff;pointer-events:none">
    Original
  </span>

  <!-- transparent range input covers the whole widget -->
  <input type="range" min="0" max="100" value="50"
    style="position:absolute;top:0;left:0;width:100%;height:100%;
           opacity:0;cursor:col-resize;margin:0;padding:0;z-index:10;
           -webkit-appearance:none;appearance:none"
    oninput="
      var p = this.value;
      document.getElementById('top_{uid}').style.clipPath  = 'inset(0 0 0 '+p+'%)';
      document.getElementById('line_{uid}').style.left     = p+'%';
      document.getElementById('hdl_{uid}').style.left      = p+'%';
    "/>
</div>
"""

# ─── Legend ──────────────────────────────────────────────────
def legend_html(present_indices):
    items = ""

    for i in sorted(present_indices):
        if i >= NUM_CLASSES:
            continue

        r, g, b = VOC_PALETTE[i]
        items += (
            f'<div style="display:flex;align-items:center;gap:5px;'
            f'font-size:10px;color:#374151">'
            f'<div style="width:11px;height:11px;border-radius:3px;flex-shrink:0;'
            f'background:rgb({r},{g},{b})"></div>'
            f'{VOC_CLASSES[i]}</div>'
        )

    return f"""
<div style="background:#fff;border-radius:10px;padding:12px;margin-top:12px;
            box-shadow:0 2px 8px rgba(0,0,0,.12)">
  <div style="font-size:9.5px;font-weight:700;color:#6b7280;
              text-transform:uppercase;letter-spacing:.1em;
              margin-bottom:8px;text-align:center">
    Detected classes
  </div>
  <div style="display:flex;flex-wrap:wrap;gap:6px;justify-content:center">
    {items}
  </div>
</div>
"""

# ─── CSS ─────────────────────────────────────────────────────
CSS = """
<style>
*, *::before, *::after {
  box-sizing:border-box;
}

.wrap {
  background:#0d0d1f;
  padding:20px;
  border-radius:16px;
}

.grid {
  display:grid;
  grid-template-columns:1fr 1fr 1fr;
  gap:20px;
  align-items:start;
}

.ptitle {
  font-size:11px;
  font-weight:700;
  letter-spacing:.12em;
  text-transform:uppercase;
  text-align:center;
  padding:5px 10px;
  border-radius:7px;
  margin-bottom:10px;
}

.t-orig, .t-unet, .t-seg {
  color:#ffffff;
  background:rgba(0,0,0,0.5);
}

.orig-img img {
  width:100%;
  height:auto;
  border-radius:10px;
  display:block;
  box-shadow:0 4px 20px rgba(0,0,0,.5);
}

@media (max-width: 900px) {
  .grid {
    grid-template-columns:1fr;
  }
}
</style>
"""

# ─── Main inference function ─────────────────────────────────
def process_image(input_img):
    if input_img is None:
        return (
            "<p style='color:#666;text-align:center;padding:40px'>"
            "Upload an image and click Run.</p>"
        )

    if isinstance(input_img, np.ndarray):
        pil_img = Image.fromarray(input_img).convert("RGB")
    else:
        pil_img = input_img.convert("RGB")

    # Preserve original image and aspect ratio for display
    orig_pil = pil_img.copy()
    original_size = orig_pil.size

    orig_b64 = pil_b64(orig_pil)

    # Model still receives 512x512 input
    tensor = preprocess_image(pil_img)

    present = set()

    # U-Net
    if unet_model is not None:
        pred_u = run_unet(tensor)
        present.update(int(x) for x in np.unique(pred_u))

        mask_u = resize_mask_to_original(pred_u, original_size)
        overlay_u = make_overlay(orig_pil, mask_u, alpha=0.6)

        unet_col = ba_widget(orig_b64, pil_b64(overlay_u))
    else:
        unet_col = (
            "<p style='color:#555;text-align:center;padding:30px'>"
            "U-Net not loaded</p>"
        )

    # SegFormer
    if segformer_model is not None:
        pred_s = run_segformer(tensor)
        present.update(int(x) for x in np.unique(pred_s))

        mask_s = resize_mask_to_original(pred_s, original_size)
        overlay_s = make_overlay(orig_pil, mask_s, alpha=0.6)

        seg_col = ba_widget(orig_b64, pil_b64(overlay_s))
    else:
        seg_col = (
            "<p style='color:#555;text-align:center;padding:30px'>"
            "SegFormer not loaded</p>"
        )

    return f"""
{CSS}
<div class="wrap">
  <div class="grid">

    <div>
      <div class="ptitle t-orig">Original</div>
      <div class="orig-img">
        <img src="data:image/png;base64,{orig_b64}"/>
      </div>
      {legend_html(present)}
    </div>

    <div>
      <div class="ptitle t-unet">U-Net · ResNet34</div>
      {unet_col}
    </div>

    <div>
      <div class="ptitle t-seg">SegFormer · mit-b0</div>
      {seg_col}
    </div>

  </div>
</div>
"""

# ─── Gradio UI ───────────────────────────────────────────────
with gr.Blocks(title="Seg Compare") as demo:
    gr.HTML("""
    <div style="text-align:center;padding:24px 0 8px;font-family:'Segoe UI',sans-serif">
      <h1 style="font-size:24px;font-weight:800;color:#e2e2f0;margin:0">
        🔍 Segmentation Model Comparison
      </h1>
      <p style="color:#64748b;font-size:13px;margin-top:6px">
        U-Net (ResNet34) &nbsp;·&nbsp; SegFormer (mit-b0) &nbsp;·&nbsp;
        Pascal VOC 2012 &nbsp;·&nbsp;
        drag ⇔ to compare original and mask overlay
      </p>
    </div>
    """)

    upload = gr.Image(label="Upload image", type="pil")
    run_btn = gr.Button("▶ Run Inference", variant="primary")
    output = gr.HTML()

    run_btn.click(
        fn=process_image,
        inputs=upload,
        outputs=output,
    )

demo.launch(share=True)